# Day 6 — PySpark: Multi-level Grouping (groupBy, rollup, cube)

> **Topics:** `groupBy().agg()` · `.rollup()` · `.cube()` · `F.grouping()` · `F.coalesce()`  
> **Run order:** top to bottom — setup cell creates the DataFrame

In [ ]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day6_GroupBy_Rollup_Cube') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# ── Sales data: region, category, product, revenue ────────────────────────────
sales_data = [
    (1,  'North', 'Electronics', 'Laptop',   '2024-01-05', 1200.0),
    (2,  'North', 'Electronics', 'Phone',    '2024-01-10',  800.0),
    (3,  'North', 'Furniture',   'Chair',    '2024-01-12',  300.0),
    (4,  'North', 'Furniture',   'Desk',     '2024-01-15',  600.0),
    (5,  'South', 'Electronics', 'Tablet',   '2024-01-08',  500.0),
    (6,  'South', 'Electronics', 'Laptop',   '2024-01-20', 1100.0),
    (7,  'South', 'Furniture',   'Sofa',     '2024-01-18',  900.0),
    (8,  'South', 'Furniture',   'Wardrobe', '2024-01-22', 1500.0),
    (9,  'East',  'Electronics', 'Phone',    '2024-01-09',  750.0),
    (10, 'East',  'Furniture',   'Chair',    '2024-01-14',  280.0),
]
sales_schema = StructType([
    StructField('sale_id',   IntegerType(), False),
    StructField('region',    StringType(),  True),
    StructField('category',  StringType(),  True),
    StructField('product',   StringType(),  True),
    StructField('sale_date', StringType(),  True),
    StructField('revenue',   DoubleType(),  True),
])
df_sales = spark.createDataFrame(sales_data, schema=sales_schema)
print('Sales rows:', df_sales.count())

In [ ]:
df_sales.show()

---
## 1. Basic groupBy — One Row per (Region, Category)

In [ ]:
df_basic = (
    df_sales
    .groupBy('region', 'category')
    .agg(
        F.count('*').alias('num_sales'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.round(F.avg('revenue'), 2).alias('avg_revenue'),
    )
    .orderBy('region', 'category')
)
print('One row per (region, category):')
df_basic.show()

---
## 2. rollup() — Hierarchical Subtotals

`.rollup('region', 'category')` produces:
1. (region, category) — detail rows
2. (region, NULL)     — subtotal per region
3. (NULL, NULL)       — grand total

NULL in output = that column was aggregated away

In [ ]:
# Raw rollup — see the NULLs
df_rollup_raw = (
    df_sales
    .rollup('region', 'category')
    .agg(F.round(F.sum('revenue'), 2).alias('total_revenue'))
    .orderBy('region', 'category')
)
print('ROLLUP raw — NULL = rolled up:')
df_rollup_raw.show()

In [ ]:
# Label NULL rows with coalesce
df_rollup = (
    df_sales
    .rollup('region', 'category')
    .agg(F.round(F.sum('revenue'), 2).alias('total_revenue'))
    .withColumn('region',   F.coalesce(F.col('region'),   F.lit('ALL REGIONS')))
    .withColumn('category', F.coalesce(F.col('category'), F.lit('ALL CATEGORIES')))
    .orderBy('region', 'category')
)
print('ROLLUP with labeled subtotals:')
df_rollup.show()

In [ ]:
# F.grouping() to detect which level each row represents
# 1 = rolled up, 0 = real value
df_rollup_level = (
    df_sales
    .rollup('region', 'category')
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.grouping('region').alias('region_rolled'),
        F.grouping('category').alias('cat_rolled'),
    )
    .withColumn('level',
        F.when((F.col('region_rolled') == 0) & (F.col('cat_rolled') == 0), 'detail')
         .when((F.col('region_rolled') == 0) & (F.col('cat_rolled') == 1), 'region subtotal')
         .otherwise('grand total')
    )
    .withColumn('region',   F.coalesce(F.col('region'),   F.lit('ALL')))
    .withColumn('category', F.coalesce(F.col('category'), F.lit('ALL')))
    .drop('region_rolled', 'cat_rolled')
    .orderBy('level', 'region', 'category')
)
print('ROLLUP with level label:')
df_rollup_level.show()

---
## 3. cube() — All Possible Combinations

`.cube('region', 'category')` produces ALL grouping combinations:
- (region, category) — detail
- (region)           — region subtotals
- (category)         — category subtotals  ← **ROLLUP doesn't give this**
- ()                 — grand total

In [ ]:
df_cube = (
    df_sales
    .cube('region', 'category')
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.grouping('region').alias('region_rolled'),
        F.grouping('category').alias('cat_rolled'),
    )
    .withColumn('level',
        F.when((F.col('region_rolled') == 0) & (F.col('cat_rolled') == 0), 'region+category')
         .when((F.col('region_rolled') == 0) & (F.col('cat_rolled') == 1), 'region only')
         .when((F.col('region_rolled') == 1) & (F.col('cat_rolled') == 0), 'category only')
         .otherwise('grand total')
    )
    .withColumn('region',   F.coalesce(F.col('region'),   F.lit('ALL')))
    .withColumn('category', F.coalesce(F.col('category'), F.lit('ALL')))
    .drop('region_rolled', 'cat_rolled')
    .orderBy('level', 'region', 'category')
)
print('CUBE — all combinations including category-only subtotals:')
df_cube.show(20)

---
## 4. ROLLUP vs CUBE — Side by Side

| Operator | Levels produced |
|----------|-----------------|
| `groupBy('a','b')` | (a, b) only |
| `rollup('a','b')` | (a,b), (a), () |
| `cube('a','b')` | (a,b), (a), (b), () |

In [ ]:
print('groupBy row count:', df_sales.groupBy('region','category').agg(F.sum('revenue')).count())
print('rollup  row count:', df_sales.rollup('region','category').agg(F.sum('revenue')).count())
print('cube    row count:', df_sales.cube('region','category').agg(F.sum('revenue')).count())

# With 3 regions and 2 categories:
# groupBy: 6 (all region×category combos)
# rollup:  6 detail + 3 region subtotals + 1 grand total = 10
# cube:    6 detail + 3 region subtotals + 2 category subtotals + 1 grand total = 12

---
## Quick Reference

```python
# Basic groupBy
df.groupBy('a', 'b').agg(F.sum('val').alias('total'))

# ROLLUP — hierarchical
df.rollup('a', 'b').agg(F.sum('val').alias('total'))

# CUBE — all combinations
df.cube('a', 'b').agg(F.sum('val').alias('total'))

# Label NULLs
.withColumn('a', F.coalesce(F.col('a'), F.lit('ALL')))

# Detect rollup level
F.grouping('col')    # 1 = rolled up, 0 = real value
```

In [ ]:
spark.stop()
print('Spark stopped.')